# Task 1 — Topological spatial relation qualification

Reproduction of the experiment done by Ji et al (2025) using Jev by TypesafeAI instead of GPT-4. The same data and evaluation methods are used. Prompts are adapted to the Jev model and different prompting strategies (instructions and criteria) are tested.

Input: Two WKT geometries
Output: Topological predicate


Ji, Y., Gao, S., Nie, Y., Majić, I., & Janowicz, K. (2025). Foundation models for geospatial reasoning: Assessing the capabilities of large language models in understanding geometries and topological spatial relations. International Journal of Geographical Information Science, 39(9), 1866–1903. https://doi.org/10.1080/13658816.2025.2511227

## Imports and functions

In [1]:
import pandas as pd
from tqdm import tqdm
from tqdm.asyncio import tqdm 
from datetime import datetime
import os
import json
import asyncio
from dotenv import load_dotenv
x = load_dotenv()

from typesafe_sdk import Choice, TypeSafeClient, AsyncTypeSafeClient

# Model dump for function create_metadata
async def model_list() -> None:
    async with AsyncTypeSafeClient() as client:
        return await client.models.list()

models = await model_list()
model_dump = models.model_dump()

TypeSafeError: No API key was provided. Pass api_key or set the TYPESAFE_API_KEY environment variable.

In [2]:
def format_duration(td):
    total_seconds = int(td.total_seconds())
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    return f"{hours:02}:{minutes:02}:{seconds:02} (h:m:s)"

def create_metadata(instructions, criteria, task, criteria_dict, instructions_dict, model_dump):
    meta = {}
    meta['Title'] = f'Task{task}_ins-{instructions}_cri-{criteria}'
    meta['Date'] = datetime.now().strftime('%Y/%m/%d %H:%M')
    meta['Task'] = task

    meta['instructions_id'] = instructions
    meta['instructions'] = instructions_dict[instructions]
    meta['criteria_id'] = criteria
    meta['criteria'] = criteria_dict[criteria]

    meta['model'] = 'jev-latest'
    meta['model_metadata'] = next((m for m in model_dump['models'] if m['name'] == meta['model']), None)

    path = f"results/{meta['Title']}.json"
    i = 0

    while os.path.exists(path):
        i += 1
        path = f"results/{meta['Title']}"
        path = f"{path}_{i}.json"

    with open(f'{path}', 'x', encoding='utf-8') as f:
        json.dump(meta, f, indent=4, ensure_ascii=False)
        print(f'Saved {path}')

    return path

def ask_jev(instructions, criteria, state):
    starttime = datetime.now()
    response = client.system_one(
        state=state,
        questions={
            "Task": Choice(
                instructions=instructions,
                criteria=criteria,
            )
        },
    )
    endtime = datetime.now()
    time = str(endtime - starttime)

    return response, time

async def async_jev(instructions, criteria, state):
    starttime = datetime.now()
    response = await client.system_one(
        state=state,
        questions={
            "Task": Choice(
                instructions=instructions,
                criteria=criteria,
            )
        },
    )
    endtime = datetime.now()
    time = str(endtime - starttime)

    return response, time

def print_summary(metadata):
    with open(metadata, 'r', encoding='utf-8') as js:
        meta = json.load(js)
    if os.path.exists(metadata.replace('.json', '.csv')):
        results_df = pd.read_csv(metadata.replace('.json', '.csv'))
    else:
        print('No results available yet.')
        return

    high_conf = results_df[results_df['confidence'].astype(float)>=0.8]
    mid_conf = results_df[(results_df['confidence'].astype(float)<0.8) & (results_df['confidence'].astype(float)>=0.5)]
    low_conf = results_df[results_df['confidence'].astype(float)<0.5]

    summary = f"""
        Summary of results for {meta['Title']}:
        Date: {meta['Date']}
        Tokens used: {int(meta['Input Tokens']) + int(meta['Output Tokens'])}
        Time taken (synchronous): {meta['Duration']}

        Total: {results_df.shape[0]} results, {round(results_df['correct'].mean()*100, 2)}% correct.
        Confidence above 80%: {high_conf.shape[0]} results, {round(high_conf['correct'].mean()*100, 2)}% correct.
        Confidence between 50 and 80%: {mid_conf.shape[0]} results, {round(mid_conf['correct'].mean()*100, 2)}% correct.
        Confidence below 50%: {low_conf.shape[0]} results, {round(low_conf['correct'].mean()*100, 2)}% correct.\n
    """
    return summary

def save_data(metadata, results_df):
    with open(metadata, 'r', encoding='utf-8') as js:
        meta = json.load(js)
    results_df.to_csv(metadata.replace('.json', '.csv'), index=False)

    meta['Input Tokens'] = int(results_df['input_tokens'].astype(int).sum())
    meta['Output Tokens'] = int(results_df['output_tokens'].astype(int).sum())
    meta['Duration'] = format_duration(pd.to_timedelta(results_df['time']).sum())
    meta['Accuracy'] = f'{round(results_df['correct'].mean()*100, 2)}%'
    meta['Results'] = results_df.shape[0]

    with open(f'{metadata}', 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=4, ensure_ascii=False)
        print(f'Saved {metadata}')

    print(print_summary(metadata))

    return metadata
    

## Execution

In [ ]:
instructions_dict = {
    'basic': "What topological predicate best describes the relationship between Place A and Place B? The predicates follow the definition of DE-9IM.",
    'think': "What topological predicate best describes the relationship between Place A and Place B? The predicates follow the definition of DE-9IM. Think step by step.", # Thinking step by step is contradictory to system one intelligence. It also did not influence results.
    'ji': "You will be given the WKT format of geometries given the subject A and reference object B. Classify their spatial relations into one of the predicates defined by DE-9IM and Open Geospatial Consortium. If A equals to B, there is no need to return 'within' or 'contains'."
}

In [17]:
criteria_dict = {
    'basic': {
        "contains": "",
        "within": "",
        "touches": "",
        "crosses": "",
        "disjoint": "",
        "overlaps": "",
        "equals": "",
    },
    'explained': {}
}

In [18]:
# Same data as in paper

relations = pd.read_csv('relations.csv')
relations_test = relations[relations['idx']<40].reset_index(drop=True)
print(relations_test.shape[0])
relations_test[['objectid', 'geom_type', 'geom_type_subject', 'predicate']].groupby(['geom_type_subject', 'predicate', 'geom_type']).count()

1400


objectid
geom_type_subject predicate geom_type           
LineString        contains  LineString        40
                            Point             40
                  crosses   LineString        40
                            Polygon           40
                  disjoint  LineString        40
                            Point             40
                            Polygon           40
                  equals    LineString        40
                  overlaps  LineString        40
                  touches   LineString        40
                            Point             40
                            Polygon           40
                  within    LineString        40
                            Polygon           40
Point             disjoint  LineString        40
                            Point             40
                            Polygon           40
                  equals    Point             40
                  touches   LineString        40
                            Polygon           40
                  within    LineString        40
                            Polygon           40
Polygon           contains  LineString        40
                            Point             40
                            Polygon           40
                  crosses   LineString        40
                  disjoint  LineString        40
                            Point             40
                            Polygon           40
                  equals    Polygon           40
                  overlaps  Polygon           40
                  touches   LineString        40
                            Point             40
                            Polygon           40
                  within    Polygon           40

In [ ]:
# ASYNCHROUNOUS EXECUTION (execution of this cell costs about 5 cent)

client = AsyncTypeSafeClient()

instructions = 'think'
criteria = 'basic'
path = create_metadata(instructions, criteria, '1', criteria_dict, instructions_dict, model_dump)

async def main():
    results = []
    tasks = []
    
    for row in relations_test.itertuples():
        state = f"Geometry A: {row.geometry}, Geometry B: {row.geometry_subject}"
        task = async_jev(instructions_dict[instructions], criteria_dict[criteria], state)
        tasks.append(task)

    results_raw = await tqdm.gather(*tasks, desc="Prompting")

    for i, (response, time) in enumerate(results_raw):
        row = relations_test.iloc[i] # Match the result to the original row
        result = {
            'GeomA': row.geometry,
            'GeomB': row.geometry_subject,
            'predicate': row.predicate,
            'choice': response.answers['Task'].choice,
            'probabilities': str(response.answers['Task'].probabilities),
            'confidence': response.answers['Task'].confidence,
            'correct': int(response.answers['Task'].choice == row.predicate),
            'input_tokens': response.usage.input_tokens,
            'output_tokens': response.usage.output_tokens,
            'time': time
        }
        results.append(result)

    results_df = pd.DataFrame(results)
    save_data(path, results_df)


await main()

Saved results/Task1_ins-think_cri-basic_1.json


Prompting: 100%|██████████| 1400/1400 [00:05<00:00, 242.29it/s]

Saved results/Task1_ins-think_cri-basic_1.json

        Summary of results for Task1_ins-think_cri-basic:
        Date: 2026/10/20 10:47
        Tokens used: 875296
        Time taken (synchronous): 01:15:59 (h:m:s)

        Total: 1400 results, 35.21% correct.
        Confidence above 80%: 355 results, 92.68% correct.
        Confidence between 50 and 80%: 370 results, 19.73% correct.
        Confidence below 50%: 675 results, 13.48% correct.

    


## Result Evaluation

I am disregarding geometry type in the experiment (as context) and in the evaluation. The paper did so also, but it might be interesting.

In [47]:
results_list = os.listdir('results')
all_results = []

for result in results_list:
    if result.endswith('.csv'):
        continue
    result = result.rstrip('.json')

    with open(f'results/{result}.json', 'r', encoding='utf-8') as js:
        meta = json.load(js)

    
    results_df = pd.read_csv(f'results/{result}.csv')

    high_conf = results_df[results_df['confidence'].astype(float)>=0.8]
    mid_conf = results_df[(results_df['confidence'].astype(float)<0.8) & (results_df['confidence'].astype(float)>=0.5)]
    low_conf = results_df[results_df['confidence'].astype(float)<0.5]

    result_dict = {
        'Title': meta['Title'],
        'Date': meta['Date'],
        'Instructions': meta['instructions_id'],
        'Criteria': meta['criteria_id'],
        'Input tokens': meta['Input Tokens'],
        'Results': results_df.shape[0],
        'Total %': round(results_df['correct'].mean()*100, 2),
        'High Conf': high_conf.shape[0],
        'High Conf %': round(high_conf['correct'].mean()*100, 2),
        'Mid Conf': mid_conf.shape[0],
        'Mid Conf %': round(mid_conf['correct'].mean()*100, 2),
        'Low Conf': low_conf.shape[0],
        'Low Conf %': round(low_conf['correct'].mean()*100, 2),
    }
    predicates = ["contains","within","touches","crosses","disjoint","overlaps","equals"]
    for pred in predicates:
        df = results_df[results_df['choice'] == pred]
        correct = round(df['correct'].mean()*100, 2)
        result_dict[f'{pred}%'] = float(correct)

    all_results.append(result_dict)

all_results_df = pd.DataFrame(all_results)
all_results_df.sort_values('Instructions')

,Title,Date,Instructions,Criteria,Input tokens,Results,Total %,High Conf,High Conf %,Mid Conf,Mid Conf %,Low Conf,Low Conf %,contains%,within%,touches%,crosses%,disjoint%,overlaps%,equals%
2,Task1_ins-basic_cri-basic,2026/10/20 10:01,basic,basic,764950,1400,36.00,354,91.24,387,20.16,659,15.63,1.49,0.32,48.72,44.44,83.33,49.09,92.11
3,Task1_ins-basic_cri-basic,2026/10/20 10:47,basic,basic,764950,1400,35.64,353,91.50,384,20.05,663,14.93,1.24,0.32,48.08,25.00,83.12,46.55,92.00
0,Task1_ins-ji_cri-basic,2026/10/20 10:47,ji,basic,816750,1400,44.00,321,87.54,507,25.44,572,36.01,0.00,20.68,66.99,14.29,84.24,35.21,89.39
4,Task1_ins-ji_cri-basic,2026/10/20 10:12,ji,basic,816750,1400,42.93,322,88.82,500,26.20,578,31.83,0.00,19.67,66.67,0.00,84.62,32.89,89.06
1,Task1_ins-think_cri-basic,2026/10/20 10:47,think,basic,771950,1400,35.21,355,92.68,370,19.73,675,13.48,0.51,0.59,49.67,40.00,83.68,45.00,92.75
5,Task1_ins-think_cri-basic,2026/10/20 10:11,think,basic,771950,1400,35.64,351,92.88,375,20.80,674,14.09,0.77,0.31,48.43,50.00,82.90,42.59,90.79


In [ ]:
# Evaluation of individual predicate accuracy

results_df = pd.read_csv(f'results/Task1_ins-ji_cri-basic.csv')


pred_results = {}
predicates = ["contains","within","touches","crosses","disjoint","overlaps","equals"]
for pred in predicates:
    df = results_df[results_df['choice'] == pred]
    correct = round(df['correct'].mean()*100, 2)
    pred_results[pred] = float(correct)

pred_results

{'contains': 0.0,
 'within': 19.67,
 'touches': 66.67,
 'crosses': 0.0,
 'disjoint': 84.62,
 'overlaps': 32.89,
 'equals': 89.06}

## Archive

In [ ]:
# SYNCHRONOUS EXECUTION (waaay slower; use asynchronous)
client = TypeSafeClient()

instructions = 'think'
criteria = 'basic'
path = create_metadata(instructions, criteria, '1', criteria_dict, instructions_dict)
results = []

df = pd.read_csv('relations.csv')
df = df[:1]

for row in tqdm(df.itertuples(), total=len(df), desc="Prompting"):
    result = {}
    state = f"Geometry A: {row.geometry}, Geometry B: {row.geometry_subject}"
    response, time = ask_jev(instructions_dict[instructions], criteria_dict[criteria], state)

    result['GeomA'] = row.geometry
    result['GeomB'] = row.geometry_subject 
    result['predicate'] = row.predicate
    result['choice'] = response.answers['Task'].choice
    result['probabilities'] = str(response.answers['Task'].probabilities)
    result['confidence'] = response.answers['Task'].confidence
    result['correct'] = int(result['choice'] == result['predicate'])
    result['input_tokens'] = response.usage.input_tokens
    result['output_tokens'] = response.usage.output_tokens
    result['time'] = time
    
    results.append(result)

results_df = pd.DataFrame(results)
save_data(path, results_df)

Saved results/Task1_ins-think_cri-basic_3.json


Processing geometries: 100%|██████████| 1/1 [00:20<00:00, 20.61s/it]

Saved results/Task1_ins-think_cri-basic_3.json

        Summary of results for Task1_ins-think_cri-basic:
        Date: 2026/10/19 17:34
        Tokens used: 506
        Time taken: 00:00:20 (h:m:s)

        Total: 1 results, 0.0% correct.
        Confidence above 80%: 1 results, 0.0% correct.
        Confidence between 50 and 80%: 0 results, nan% correct.
        Confidence below 50%: 0 results, nan% correct.

    


'results/Task1_ins-think_cri-basic_3.json'